# 2. What is a PSF calibration?
**Before you start:** select the **Python (DECODE Intel)** kernel. Run each cell from top to bottom (Shift+Enter). If you change `config.yaml`, rerun the first cell to reload it. Nothing is saved unless a cell tells you it will save.
**Goal:** understand the file needed to simulate *your* microscope. No fitting is performed by this notebook.

A PSF is the shape of the spot a point-like emitter makes on the camera. A bead z-stack records that spot at different depths. SMAP fits a *cubic spline* to the stack; DECODE can then use the resulting `3dcal.mat` to simulate spots.

**What to ask the data collector:** “Is there a bead z-stack or SMAP 3D spline calibration for each channel, measured with the same optical setup as my recording? What are the channel, z-step, orientation, camera offset, gain and exposure?” If no matching calibration exists, ask whether the lab can acquire one and whether it represents the original experiment.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import yaml

# Start Jupyter from the project folder or decode_walkthrough/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'decode_walkthrough' / 'config.yaml').is_file():
    ROOT = ROOT.parent
config_path = ROOT / 'decode_walkthrough' / 'config.yaml'
if not config_path.is_file():
    raise FileNotFoundError('Start Jupyter in the project folder or decode_walkthrough/.')
with config_path.open() as f:
    CFG = yaml.safe_load(f)
def project_path(value):
    if value is None: return None
    p = Path(value).expanduser()
    return p if p.is_absolute() else ROOT / p
EXAMPLE = project_path(CFG['paths']['example_folder'])
WORK = project_path(CFG['paths']['work_folder'])
print('Project:', ROOT, '| Config:', config_path, '| Example exists:', EXAMPLE.exists())


## A. Inspect DECODE’s example bead stack
This stack is DECODE’s example. It is not your calibration. Compare the three depth slices; the apparent shapes and brightness can change with focus.

In [ ]:
bead_files = sorted((EXAMPLE / 'beads').glob('*/*.tif'))
print('Example bead stacks:', len(bead_files))
if not bead_files: raise FileNotFoundError('Copy the full experimental_data_workflow example folder to ROOT.')
beads_path = bead_files[0]
with tifffile.TiffFile(beads_path) as t:
    n = len(t.pages); print('File:', beads_path, '| pages:', n, '| first page:', t.pages[0].shape)
    images = [t.pages[i].asarray() for i in (0, n//2, n-1)]
fig, axes = plt.subplots(1,3,figsize=(14,4))
for ax, image, ix in zip(axes, images, (0,n//2,n-1)):
    ax.imshow(image, cmap='gray'); ax.set_title(f'z-plane {ix} (auto contrast)')
plt.show()


## B. Inspect the fitted example file
A `.mat` file can contain spline coefficients. Listing its contents only confirms the file can be read; it does not check fit quality.

In [ ]:
from scipy.io import whosmat
calibration = project_path(CFG['paths']['example_calibration'])
print('Example calibration contents:', whosmat(calibration))
print('This file is an existing fit; inspecting it does not calibrate your microscope.')


## C. Obtain a calibration for your data
1. Open the relevant bead z-stack in SMAP on a suitable computer.
2. Use **Analyze / sr3D / calibrate3DsplinePSF**. Supply the measured z step and settings from the acquisition. For 2D data DECODE’s authors recommend a bidirectional fit.
3. Inspect fit quality in SMAP, save `3dcal.mat`, and record which channel and setup it represents. Repeat for the other channel if appropriate.
4. Record the paths under `own_data.channels` in `config.yaml`. **Those fields are notes only:** our example training notebooks do not automatically use them. Adapting training requires a matching DECODE parameter file.

[DECODE’s calibration instructions](https://decode.readthedocs.io/en/release-0.10/tutorial.html) include SMAP links. Without a matching calibration, use the bundled files to learn, not to draw conclusions about your own microscope.
